# Week 10 — does the wiring matter, on fresh seeds, and the last 4 seeds

## Where week 9 left it
Frozen male-CNS connectome + readout **C4** (3,000 neurons re-chosen on all DAgger data, ridge, release region up-weighted to 1/3 of the fit): **46/50** on seeds 92000–92049 (95% CI 81–97%). Pass (≥ 40/50): yes. Failures: 3 hover above the target still holding the cube (the press, which C4 traded away), 1 never lifted. No-brain controls: linear on the 8 signals 0/50, MLP 50/50.

## This notebook (everything tested on **fresh seeds 93000–93049**, never used before)
| step | what | GPU time |
|---|---|---|
| 1 | no-brain controls (CPU): expert; MLP; linear; linear with C4's weights; linear with C6's weights; **gated linear (G)**, all on the 8 noise-free signals, trained like the brain | 0 (~10 min CPU) |
| 2 | **connectome, C4**: week 9's readout, refitted from week 9's own DAgger data (attach week 9's output), tested on the fresh seeds | ~5 min |
| 3 | **random network, C4**: its own active neurons, its own 3,000-neuron readout, the same recipe, seeds and timing | ~45 min |
| 4 | connectome, G: grip-gated readout (two linear motion maps, switched by the decoded grip). **Off by default** (`RUN_G`): its no-brain control already passes (below) | (~40 min) |
| 5 | **connectome, C6**: C4 + the press rows up-weighted to 1/4 of the fit (still linear) | ~40 min |
| 6 | random network, G (`RUN_RANDOM_G`, off by default) | (~45 min) |

**Sandbox result for step 1, already in (the physics is the same on Kaggle):** on the fresh seeds, the **gated linear readout on the 8 noise-free signals completes 41/50 with no brain at all** (linear 0/50, linear with C4's weights 0/50, with C6's weights 0/50, MLP 50/50). So the gate itself does most of the task, and a connectome-G result could not show that the brain is needed (week 7's reason for not using an MLP). G is kept in the code, off by default. Default run: steps 1, 2, 3, 5 (~1.5 h of GPU).

Training is week 7–9's: DAgger 4 × 30 on seeds 64000+, round 0 expert-driven, half of each round with 5 mm action noise; 80 ms of brain per 40 ms step, 80 ms window. Steps 4–5 go straight to the closed loop (no offline step).

## Rules (fixed before the run; `week10_plan.md`)
1. **Headline:** connectome C4 on the fresh seeds. Pass ≥ 40/50 (unchanged since week 7).
2. **Wiring (pre-registered, week 7's rule):** connectome C4 vs random network C4, paired by seed, milestone by milestone from the hardest (success → carried to zone → lifted); the first with exact McNemar p < 0.05 decides. Placement and time compared (Mann–Whitney) only if both have ≥ 10 successes. Repeated for G as a secondary result if step 6 runs.
3. **Toward 50/50:** each readout's successes on the same seeds, paired vs C4. Best = most successes (ties → simpler: C4, C6, G). **A readout's result counts as the brain's only if its own no-brain control stays below 40/50.** 50/50 reached if the best readout is 50/50.

Needs (Kaggle inputs, or local folders; see the README's *Reproducing* section): this repo (attached as a dataset, or cloned by the setup cell), the data folder from `scripts/fetch_connectome.py`, and **week 9's notebook output** (for step 2; without it the connectome C4 is retrained, ~40 min more).

### Setup

In [ ]:
import sys, subprocess, os, shutil, zipfile, re
from pathlib import Path

# ---- where are we? -------------------------------------------------------------------------------
# Kaggle is checked first: its image also ships the google.colab package, but can't mount Drive.
KAGGLE_INPUT = Path('/kaggle/input')
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or KAGGLE_INPUT.is_dir()
IN_COLAB = not IN_KAGGLE and Path('/var/colab/hostname').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
HOST = 'colab' if IN_COLAB else 'kaggle' if IN_KAGGLE else 'laptop'
if IN_COLAB or IN_KAGGLE:
    os.environ.setdefault('MUJOCO_GL', 'egl')                      # headless video; set before mujoco loads
    os.environ.setdefault('NUMBA_CACHE_DIR', '/tmp/numba_cache')   # the code folder is read-only on Kaggle

# ---- packages ------------------------------------------------------------------------------------
PKGS = ['brian2', 'numba', 'mujoco', 'imageio', 'imageio-ffmpeg']
if IN_COLAB or IN_KAGGLE:
    try:
        import brian2, numba, mujoco, imageio_ffmpeg
    except ImportError:
        if IN_KAGGLE:
            # A batch run (Save & Run All) can't survive a kernel restart, so numpy is pinned to the
            # installed version: nothing gets upgraded, nothing needs a restart.
            import numpy
            r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS, f'numpy=={numpy.__version__}'])
            if r.returncode:
                raise RuntimeError("pip install failed. Notebook settings (right panel) -> Internet: On. "
                                   "Kaggle only offers Internet to phone-verified accounts.")
            import brian2, numba, mujoco, imageio_ffmpeg
        else:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS], check=True)
            print("packages installed -> restarting runtime. When it reconnects, choose Runtime -> Run all again.")
            os.kill(os.getpid(), 9)

# ---- folders -------------------------------------------------------------------------------------
# Kaggle: the inputs are read-only and can come from several datasets and earlier notebook versions.
# They are linked into one tree with the repo layout (sim/, arm/, data/, mujoco_menagerie/), so every
# path below is the same as in a local checkout. Results go to /kaggle/working.
def _make_real(d):
    """Make folder d (under ROOT) a real folder. Any part of its path that is a link into a read-only input
    becomes a real folder holding links to that input's contents, so a second input can add to it."""
    p = ROOT
    p.mkdir(parents=True, exist_ok=True)
    for name in Path(d).relative_to(ROOT).parts:
        p = p / name
        if p.is_symlink() and p.is_dir():
            t = p.resolve(); p.unlink(); p.mkdir()
            for k in sorted(t.iterdir()):
                (p / k.name).symlink_to(k)
        elif not p.exists():
            p.mkdir()

def _link_tree(src, dst):
    """Link everything under src into dst, folder by folder, so several inputs can fill one tree."""
    _make_real(dst)
    for s in sorted(src.iterdir()):
        d = dst / s.name
        if d.is_symlink() and s.is_dir() and d.is_dir():   # a folder another input already linked: split it
            t = d.resolve(); d.unlink(); _link_tree(t, d)
        if s.is_dir() and d.is_dir() and not d.is_symlink():
            _link_tree(s, d)
        elif not d.exists() and not d.is_symlink():
            d.symlink_to(s)

_RUN = re.compile(r'week\d+[a-z]*(_[a-z]+)?')        # week7, week8b, week8b_quick: a notebook's results folder

def _find_roots(top, depth=4):
    """('tree', p): p holds sim/, arm/ or mujoco_menagerie/ (the repo, uploaded as a dataset).
       ('data', p): p is the data folder itself (weights_*/, wsyn_choice.json).
       ('runs', p): p holds week*/ folders (an earlier notebook version's output)."""
    found = []
    def walk(p, d):
        try:
            kids = sorted(p.iterdir())
        except OSError:
            return
        names = {k.name for k in kids}
        zips = [z for z in kids if z.suffix == '.zip' and z.stem in ('sim', 'arm', 'mujoco_menagerie')
                and z.stem not in names]           # in case an upload kept its folders zipped
        if zips:
            out = Path('/tmp/kaggle_unzip') / p.name
            for z in zips:
                if not (out / z.stem).exists():
                    zf = zipfile.ZipFile(z)
                    inside = all(n.startswith(z.stem + '/') for n in zf.namelist())
                    zf.extractall(out if inside else out / z.stem)
            found.append(('tree', out))
        if d > 0 and names & {'sim', 'arm', 'mujoco_menagerie'}:
            found.append(('tree', p))
        elif d > 0 and ('wsyn_choice.json' in names or any(n.startswith('weights_') for n in names)):
            found.append(('data', p))              # brain/data itself (an upload can drop the brain/ level)
        elif d > 0 and any(k.is_dir() and _RUN.fullmatch(k.name) for k in kids):
            found.append(('runs', p))
        elif d < depth:
            for k in kids:
                if k.is_dir():
                    walk(k, d + 1)
    walk(top, 0)
    return found

REPO_URL = 'https://github.com/Notlord69/fly-connectome-arm'   # used on Kaggle only if the repo isn't attached
if IN_KAGGLE:
    ROOT = Path('/tmp/fly-connectome-arm')                 # links only, rebuilt every session
    shutil.rmtree(ROOT, ignore_errors=True)
    ROOTS = _find_roots(KAGGLE_INPUT)
    for kind, p in sorted(ROOTS, key=lambda r: r[0] != 'runs'):   # notebook outputs first: newest results win
        if kind == 'tree':
            _link_tree(p, ROOT)
        elif kind == 'data':
            _link_tree(p, ROOT / 'data')
        else:
            for k in sorted(p.iterdir()):
                if k.is_dir() and _RUN.fullmatch(k.name):
                    _link_tree(k, ROOT / 'data' / k.name)
    print("Kaggle inputs:", *[f"{k:5s} {p}" for k, p in ROOTS], sep="\n  ")
    if not (ROOT / 'sim' / 'week10.py').exists():         # the code isn't attached: clone it
        src = Path('/tmp/fly-connectome-arm-src')
        if not src.exists():
            subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO_URL, str(src)], check=True)
        _link_tree(src, ROOT)
    CODE = ROOT
    if not (ROOT / 'mujoco_menagerie' / 'franka_emika_panda').exists():
        subprocess.run(['bash', str(CODE / 'scripts' / 'fetch_panda_model.sh'), '/tmp/mujoco_menagerie'], check=True)
        os.environ['FLYARM_MENAGERIE'] = '/tmp/mujoco_menagerie'
    DATA = ROOT / 'data'
elif IN_COLAB:
    CODE = Path('/content/drive/MyDrive/fly-connectome-arm')
    DATA = Path(os.environ.get('FLYARM_DATA_DIR', CODE / 'data'))
else:                                                       # a local checkout: this notebook is in notebooks/
    CODE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'sim' / 'week10.py').exists())
    DATA = Path(os.environ.get('FLYARM_DATA_DIR', CODE / 'data'))
os.environ['FLYARM_DATA_DIR'] = str(DATA)
RUNS = Path('/kaggle/working') if IN_KAGGLE else DATA      # where this notebook's results go
ARM = CODE / 'arm' if (CODE / 'arm' / 'panda_env.py').exists() else None
sys.path.insert(0, str(CODE))
if ARM: sys.path.insert(0, str(ARM))

def out_dir(name):
    """This notebook's results folder. On Kaggle it is /kaggle/working/<name>, saved with the version; if an
    earlier version's output is attached as an input, its <name>/ is copied in first, so RESUME carries on."""
    o = RUNS / name
    if RUNS != DATA and not o.exists() and (DATA / name).exists():
        shutil.copytree(DATA / name, o)
        print(f"resuming: copied the earlier run's {name}/ into {o}")
    o.mkdir(parents=True, exist_ok=True)
    return o


# ---- week 10 ------------------------------------------------------------------------------------
QUICK = False            # True: smoke test (DAgger 3 x 6, 10 test seeds), results in week10_quick/
RESUME = True
RUN_REFERENCES = True    # the no-brain controls (CPU, ~10 min)
RUN_RANDOM_C4 = True     # the pre-registered wiring comparison
RUN_G = False            # connectome, grip-gated readout. OFF: its no-brain control (gated linear on the
                         # 8 signals) already does 41/50 in the sandbox, so a brain result can't show the brain is needed
RUN_C6 = True            # connectome, C4 + press weighting
RUN_RANDOM_G = False     # random network, gated readout (secondary; ~45 min). OFF for the same reason
K_WIDE = 3000
USE_GPU = True
GPU_DTYPE = 'float32'
MAX_BATCH = 64

GPU = None
if USE_GPU:
    try:
        import torch
        if torch.cuda.is_available():
            GPU = f"{torch.cuda.get_device_name(0)} (x{torch.cuda.device_count()})"
            print(f"GPU: {GPU}; torch {torch.__version__}")
        else:
            print("USE_GPU = True but no GPU. Kaggle: Session options -> Accelerator -> GPU T4 x2.")
    except ImportError:
        print("torch is not installed here -> numba on the CPU")

for p in ['sim/batch7.py', 'sim/network7.py', 'sim/interface.py', 'sim/expert8.py', 'sim/expert9.py',
          'sim/week8.py', 'sim/pool8.py', 'sim/week9.py', 'sim/week10.py', 'sim/video7.py',
          'data/weights_male-cns_v1.0/meta.json', 'data/shortlist_week1.csv', 'data/wsyn_choice.json',
          'data/week7/week7_random_network_meta.json']:
    print(f"{p:50s} {'ok' if ((CODE / p) if p.startswith('sim/') else (DATA / p.split('/', 1)[1])).exists() else 'MISSING'}")
print(f"{'week 4 arm code':50s} {ARM if ARM else 'MISSING'}")
print(f"{'week 9 output (data/week9/)':50s} {'found' if (DATA / 'week9').is_dir() else 'not attached: connectome C4 will be retrained'}")

### Load the connectome, the neuron groups and the arm (as weeks 8b–9)

In [ ]:
import json, time, gc, pickle
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings('ignore', module='brian2')
from sim.lif import WeightSet, peak_mem_gb
from sim.pickplace import SIGNALS8
from sim.interface import Encoder
from sim.batch7 import make_engine, BatchBrain
from sim.network7 import wiring_labels, random_network
from sim.week7 import reference_states, comparison_rows
from sim.expert8 import expert8
from sim.pool8 import active_neurons
from sim.week10 import (RECIPES, TEST10, CUT10, week9_warm, run_network10, run_references10, compare10,
                        describe10, REFERENCES)
from panda_env import PandaPickPlace, OBS_NAMES

T_START = time.perf_counter()
OUT = out_dir('week10_quick' if QUICK else 'week10')

ws = WeightSet(DATA / 'weights_male-cns_v1.0')
W = json.loads((DATA / 'wsyn_choice.json').read_text())['w_scale']
W_RAND = json.loads((DATA / 'week7' / 'week7_random_network_meta.json').read_text())['w_scale']
JO = ws.by_type(r'^JO-(C|E)', cls='cb_sensory')
shortlist = pd.read_csv(DATA / 'shortlist_week1.csv')
g2i = pd.Series(ws.neurons.index, index=ws.neurons.bodyId)
DN_cand = g2i.loc[shortlist.query("group == 'middle_descending'").bodyId].to_numpy()
ALL_DN = ws.idx(ws.neurons['cls'].astype(str).str.contains('descending', case=False))
full = ws.subnetwork(np.arange(ws.N))
ws._out = None; gc.collect()
env = PandaPickPlace()

N_ROUNDS = 3 if QUICK else 4
N_PER_ROUND = 6 if QUICK else 30
CFG = dict(
    train_seeds=[list(range(64_000 + 1000 * r, 64_000 + 1000 * r + N_PER_ROUND)) for r in range(N_ROUNDS)],  # = weeks 7-9
    betas=[1.0] + [0.0] * (N_ROUNDS - 1),
    test_seeds=TEST10[:10] if QUICK else TEST10,                                                         # fresh
    cut_seeds=CUT10[:3] if QUICK else CUT10,
    k=K_WIDE, n_video=1, use_gpu=GPU is not None, dtype=GPU_DTYPE, max_batch=MAX_BATCH, cpu_batch=1)
print(f"male CNS: {ws.N:,} neurons; JO-C/E {len(JO)}; DNs {len(ALL_DN)}; w_scale connectome {W}, random {W_RAND:.3f}")
print(f"DAgger {N_ROUNDS} x {N_PER_ROUND}; test seeds {CFG['test_seeds'][0]}-{CFG['test_seeds'][-1]} (fresh); results in {OUT}")

def wide_setup(sub, w_scale, label):
    # week 8b's rule: every non-input neuron firing >= 1 Hz during one expert episode, + all DNs
    keep = np.asarray(sub['keep'])
    enc = Encoder(JO, wiring_labels(sub, JO, 16), SIGNALS8, kind='wiring')
    eng = make_engine(sub, JO, GPU is not None, GPU_DTYPE)
    others = np.setdiff1d(np.arange(len(keep)), np.searchsorted(keep, JO))
    s_ep = reference_states(env, expert8, OBS_NAMES, n=60)
    t0 = time.perf_counter()
    act_local, _ = active_neurons(BatchBrain(eng, enc, ALL_DN), w_scale, s_ep, others)
    active = keep[act_local]
    print(f"{label}: {len(active):,} non-input neurons fire >= 1 Hz during an expert episode "
          f"({len(active) / len(others):.1%}); {np.isin(ALL_DN, active).sum()} of them are DNs  "
          f"[{time.perf_counter() - t0:.0f} s]")
    del eng; gc.collect()
    return np.union1d(active, ALL_DN)

WARM = week9_warm(DATA / 'week9', K_WIDE)
if WARM:
    POOL_CONN = WARM['record'][WARM['pool_cols']]          # week 9's pool exactly: active neurons + all DNs
    print(f"week 9's run found: '{WARM['name']}' ({WARM['pkl'].name}), {len(POOL_CONN):,} neurons in its pool; "
          f"week 9 test: {(WARM['summary'] or {}).get('success')}/{(WARM['summary'] or {}).get('n')}")
else:
    print("week 9's output not attached: the connectome's pool is found again and C4 is retrained (~40 min more)")
    POOL_CONN = wide_setup(full, W, 'connectome')
NETS = {}

## Step 1: no-brain controls (CPU, ~10 min)
Each week 10 readout on the 8 noise-free signals, trained exactly like the brain. If one of these reaches 40/50, that readout's brain result no longer shows the brain is needed.

In [ ]:
REFS = run_references10(env, OBS_NAMES, CFG, OUT, resume=RESUME) if RUN_REFERENCES else {}
if REFS:
    display(pd.DataFrame(comparison_rows({k: v['summary'] for k, v in REFS.items()})).set_index('policy')
            .drop(columns=['success rate']))

## Step 2: connectome, C4 on fresh seeds (~5 min with week 9's data)

In [ ]:
NETS['connectome, C4'] = run_network10('connectome, C4', full, JO, POOL_CONN, env, OBS_NAMES, CFG, OUT, W, 'C4',
                                       resume=RESUME, extra_record=DN_cand, warm=WARM)
s = NETS['connectome, C4']['summary']
print(f"\nCONNECTOME, C4 (fresh seeds): {s['success']}/{s['n']} (95% CI {s['ci'][0]:.0%}-{s['ci'][1]:.0%}), "
      f"placement {s['place_mm_success']} mm, time {s['time_s']} s, {s['outcomes']}")
print("feedback cut:", NETS['connectome, C4']['cut']['summary']['outcomes'])
print(f"peak memory {peak_mem_gb():.2f} GB")

## Step 3: random network, C4 (~45 min) — the pre-registered wiring comparison

In [ ]:
RNET = None
if RUN_RANDOM_C4 or RUN_RANDOM_G:
    RNET = random_network(full, seed=0)                       # week 7's shuffle, seed 0
    POOL_RAND = wide_setup(RNET, W_RAND, 'random network')
if RUN_RANDOM_C4:
    NETS['random network, C4'] = run_network10('random network, C4', RNET, JO, POOL_RAND, env, OBS_NAMES, CFG, OUT,
                                               W_RAND, 'C4', resume=RESUME, extra_record=DN_cand)
    s = NETS['random network, C4']['summary']
    print(f"\nRANDOM NETWORK, C4: {s['success']}/{s['n']}, placement {s['place_mm_success']} mm, {s['outcomes']}")
    print("while collecting, by round:", [f"{r['success']:.0%}" for r in NETS['random network, C4']['dagger']['log']])
    print(*[l for l in describe10(compare10(NETS, REFS)) if l.startswith(('WIRING', '   '))], sep='\n')

## Step 4: connectome, G — grip-gated readout (off by default: its no-brain control passes)

In [ ]:
if RUN_G:
    NETS['connectome, G'] = run_network10('connectome, G', full, JO, POOL_CONN, env, OBS_NAMES, CFG, OUT, W, 'G',
                                          resume=RESUME, extra_record=DN_cand)
    s = NETS['connectome, G']['summary']
    print(f"\nCONNECTOME, G: {s['success']}/{s['n']} (95% CI {s['ci'][0]:.0%}-{s['ci'][1]:.0%}), "
          f"placement {s['place_mm_success']} mm, time {s['time_s']} s, {s['outcomes']}")
    print("while collecting, by round:", [f"{r['success']:.0%}" for r in NETS['connectome, G']['dagger']['log']])

## Step 5: connectome, C6 — C4 + press weighting, still linear (~40 min)

In [ ]:
if RUN_C6:
    NETS['connectome, C6'] = run_network10('connectome, C6', full, JO, POOL_CONN, env, OBS_NAMES, CFG, OUT, W, 'C6',
                                           resume=RESUME, extra_record=DN_cand)
    s = NETS['connectome, C6']['summary']
    print(f"\nCONNECTOME, C6: {s['success']}/{s['n']} (95% CI {s['ci'][0]:.0%}-{s['ci'][1]:.0%}), "
          f"placement {s['place_mm_success']} mm, time {s['time_s']} s, {s['outcomes']}")

## Step 6: random network, G (off by default)

In [ ]:
if RUN_RANDOM_G:
    NETS['random network, G'] = run_network10('random network, G', RNET, JO, POOL_RAND, env, OBS_NAMES, CFG, OUT,
                                              W_RAND, 'G', resume=RESUME, extra_record=DN_cand)
    s = NETS['random network, G']['summary']
    print(f"\nRANDOM NETWORK, G: {s['success']}/{s['n']}, placement {s['place_mm_success']} mm, {s['outcomes']}")
del RNET; gc.collect()

## Step 7: the comparison (deliverable), video, results file

In [ ]:
SUMM = {k: v['summary'] for k, v in NETS.items()}
SUMM.update({k: v['summary'] for k, v in REFS.items()})
TABLE = pd.DataFrame(comparison_rows(SUMM)).set_index('policy')
display(TABLE.style.format({c: '{:.0%}' for c in ('success rate', 'lifted', 'carried to zone', 'stalled')}, na_rep='-'))
display(pd.DataFrame({k: s['outcomes'] for k, s in SUMM.items()}).fillna(0).astype(int).T)
CMP = compare10(NETS, REFS)
for line in describe10(CMP):
    print(line)
for k, v in NETS.items():
    print(f"{k}: feedback cut {v['cut']['summary']['outcomes']}; grip agreement {v['grip_agreement']}; "
          f"hovering failures {v['hover']}; retreat-trap failures {v['trap']}")

VIDEO = None
EPISODES = [e for k in NETS for e in NETS[k]['video_eps']]
for e in EPISODES:
    print(f"video: {e['net']}, seed {e['seed']}: {e['outcome']}")
pickle.dump(EPISODES, open(OUT / 'week10_episodes.pkl', 'wb'))
from sim.video7 import render
VIDEO = OUT / f'week10_pick_place_{HOST}.mp4'
if render(str(OUT / 'week10_episodes.pkl'), str(VIDEO), slow=0.5):
    from IPython.display import Video
    display(Video(str(VIDEO), embed=True, width=900))
else:
    VIDEO = None

summary = dict(host=HOST, quick=QUICK, gpu=GPU, test_seeds=[CFG['test_seeds'][0], CFG['test_seeds'][-1]],
               warm_from_week9=None if not WARM else str(WARM['pkl']), recipes={k: v['what'] for k, v in RECIPES.items()},
               table=TABLE.reset_index().to_dict('records'), outcomes={k: s['outcomes'] for k, s in SUMM.items()},
               comparison=CMP, lines=describe10(CMP),
               networks={k: dict(recipe=v['recipe'], dagger_log=v['dagger']['log'], cut=v['cut']['summary'],
                                 hover=v['hover'], trap=v['trap'], grip_agreement=v['grip_agreement'],
                                 warm_check=v.get('warm_check'), pool=len(v['pool_cols']),
                                 per_seed={o['seed']: o['outcome'] for o in v['outs']}, wall_min=v['wall_min'])
                         for k, v in NETS.items()},
               references={k: dict(dagger_log=v['log'], per_seed={o['seed']: o['outcome'] for o in v['outs']})
                           for k, v in REFS.items()},
               wall_min=(time.perf_counter() - T_START) / 60)
(OUT / f'week10_results_{HOST}.json').write_text(json.dumps(summary, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o)))
print(f"this session {summary['wall_min']:.0f} min. Send me week10_results_{HOST}.json"
      f"{' and the video' if VIDEO else ''} from {OUT}.")